In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DISTRIBUTION = "HIERARCHICAL_PAIRS"

In [ ]:
df = pd.read_parquet("data/standard_targeted/results.parquet")

In [ ]:
# Best F1 per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best = group.loc[group["f1_score"].idxmax()]
    print(f"{bench}: F1={best['f1_score']:.4f} ")

In [ ]:
df.loc[DISTRIBUTION]

## Focus distribution

In [ ]:
sub = df.loc[DISTRIBUTION].reset_index()
sub.head()

## Marginal effects on F1 score

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["By L1 Coefficient", "By SAE L0 (effective sparsity)"],
)

fig.add_trace(
    go.Box(
        x=sub["l1_coefficient"].astype(str),
        y=sub["f1_score"],
        boxpoints="all",
        jitter=0.3,
        pointpos=0,
        marker_color=px.colors.qualitative.Plotly[0],
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=sub["sae_l0"],
        y=sub["f1_score"],
        mode="markers+text",
        text=sub["l1_coefficient"].astype(str),
        textposition="top center",
        marker=dict(size=10, color=px.colors.qualitative.Plotly[1]),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    height=450,
    width=900,
    title_text=f"F1 Score by L1 Coefficient and Effective Sparsity ({DISTRIBUTION})",
    showlegend=False,
)
fig.show()

## F1 & MCC vs sae_l0

In [ ]:
melted = sub.melt(
    id_vars=["sae_l0", "l1_coefficient"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
)

fig = px.scatter(
    melted,
    x="sae_l0",
    y="score",
    color="metric",
    facet_col="metric",
    category_orders={"metric": ["f1_score", "mcc"]},
    hover_data=["l1_coefficient"],
    labels={
        "sae_l0": "SAE L0",
        "score": "Score",
        "metric": "Metric",
    },
    title=f"F1 & MCC vs SAE L0 ({DISTRIBUTION})",
    height=400,
    width=900,
)
for i in range(1, 3):
    fig.add_vline(
        x=sub["true_l0"].mean(),
        line_dash="dash",
        line_color="gray",
        annotation_text="true L0",
        row=1,
        col=i,
    )
fig.update_traces(marker_size=10)
fig.update_layout(showlegend=False)
fig.show()

## Precision vs Recall

In [ ]:
fig = px.scatter(
    sub,
    x="recall",
    y="precision",
    size="sae_l0",
    text="l1_coefficient",
    hover_data=["sae_l0", "f1_score", "mcc"],
    labels={
        "precision": "Precision",
        "recall": "Recall",
        "sae_l0": "SAE L0",
        "l1_coefficient": "L1 Coeff",
    },
    title=f"Precision vs Recall (size = SAE L0, label = L1 coeff) ({DISTRIBUTION})",
    height=450,
    width=700,
)
fig.update_traces(marker=dict(sizemin=5), textposition="top center")
fig.show()

## F1 & MCC vs l1_coefficient

In [ ]:
melted = sub.melt(
    id_vars=["l1_coefficient"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="l1_coefficient",
    y="score",
    color="metric",
    facet_col="metric",
    category_orders={"metric": ["f1_score", "mcc"]},
    markers=True,
    labels={
        "l1_coefficient": "L1 Coefficient",
        "score": "Score",
        "metric": "Metric",
    },
    title=f"F1 & MCC vs L1 Coefficient ({DISTRIBUTION})",
    height=400,
    width=900,
)
fig.update_layout(showlegend=False)
fig.show()